# 01c · High-quality real photos (FFHQ, CelebA-HQ), synthetic faces (StyleGAN, diffusion) + landmark cache

The v1 detector called sharp real photos FAKE and blurry real photos FAKE: it only ever saw FF++'s YouTube
frames as "real". This notebook adds:

| Folder | Source | Label | Used by |
|---|---|---|---|
| `data_hq/ffhq` | FFHQ (the real half of *140k Real and Fake Faces*) | real | CNNs (real + self-blended) and CLIP |
| `data_hq/celebahq` | CelebA-HQ | real | CNNs (real + self-blended) and CLIP |
| `data_hq/stylegan` | StyleGAN (the fake half of *140k Real and Fake Faces*) | fake | CLIP probe |
| `data_hq/diffusion` | a diffusion-generated face set | fake | CLIP probe |

Every image is re-cropped with **the same YuNet detector and 1.3× square crop** as the video datasets, so all
sources are framed alike (otherwise a model could tell "FFHQ" from "FF++" by framing alone). Each source is split
80/10/10.

It then caches **MediaPipe face-mesh landmarks** for every real train/val crop of **every** dataset (FF++,
Celeb-DF, FFHQ, CelebA-HQ): 02a turns those faces into **self-blended fakes** on the fly.

**Notebook settings (right sidebar)**
- **Accelerator:** None (CPU) · **Internet:** On
- **Inputs:**
  - *140k Real and Fake Faces* (Kaggle dataset `xhlulu/140k-real-and-fake-faces`): FFHQ + StyleGAN
  - a **CelebA-HQ** image dataset (any Kaggle mirror)
  - a **diffusion-generated faces** dataset (any Kaggle set of diffusion face images)
  - the outputs of **01a** (FF++) and **01b** (Celeb-DF), for the landmark cache

Check the globs in **Settings** against the attached datasets; a source whose glob matches nothing is skipped.

**Produces:** `/kaggle/working/data_hq/<source>/` (crops + `manifest.csv`) and `/kaggle/working/landmarks/<dataset>.npz`.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"
!pip install -q --no-deps "mediapipe>=1.2"   # face mesh for the landmark cache (no-deps: keeps opencv-python)

In [ ]:
from src.kaggle_utils import setup, find_datasets
setup(data=False)

import glob, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

## Settings
`MAX_IMAGES` caps each source (a random, seeded sample). Edit the globs to match the attached datasets
(`**` searches sub-folders). `CELEBAHQ_IDENTITY_FILE` (optional) is a text file of `<image> <identity>` lines;
with it, CelebA-HQ is split by person instead of by image.

In [ ]:
MAX_IMAGES = 20000
SOURCES = {   # name: (label, glob)
    "FFHQ":      ("real", "/kaggle/input/**/real_vs_fake/real-vs-fake/*/real/*.jpg"),
    "StyleGAN":  ("fake", "/kaggle/input/**/real_vs_fake/real-vs-fake/*/fake/*.jpg"),
    "CelebAHQ":  ("real", "/kaggle/input/*celeba*hq*/**/*.jpg"),
    "Diffusion": ("fake", "/kaggle/input/*diffusion*/**/*.*"),
}
CELEBAHQ_IDENTITY_FILE = None

found = {}
for name, (label, pattern) in SOURCES.items():
    n = len(glob.glob(pattern, recursive=True))
    print(f"{name:10} {label:4} {n:7d} files  {pattern}")
    if n:
        found[name] = (label, pattern)
    else:
        print(f"           -> SKIPPED: no files match; fix the glob or attach the dataset")

## Crop every source

In [ ]:
args = " ".join(f'--source "{n}={lab}:{pat}"' for n, (lab, pat) in found.items())
if CELEBAHQ_IDENTITY_FILE and "CelebAHQ" in found:
    args += f' --identity-file "CelebAHQ={CELEBAHQ_IDENTITY_FILE}"'
!TQDM_MININTERVAL=60 python -m src.classifier.image_extract --out /kaggle/working/data_hq --max-images {MAX_IMAGES} --workers 4 {args}

In [ ]:
HQ = Path("/kaggle/working/data_hq")
counts = []
for man in sorted(HQ.glob("*/manifest.csv")):
    m = pd.read_csv(man)
    counts.append(m.groupby("split").size().rename(m.dataset.iloc[0]))
display(pd.DataFrame(counts).fillna(0).astype(int).style.set_caption("crops per split"))

fig, axes = plt.subplots(len(counts), 8, figsize=(16, 2.2 * len(counts)), squeeze=False)
for row, man in zip(axes, sorted(HQ.glob("*/manifest.csv"))):
    m = pd.read_csv(man)
    for ax, r in zip(row, m.sample(min(8, len(m)), random_state=0).itertuples()):
        ax.imshow(Image.open(man.parent / r.split / r.label / r.file)); ax.axis("off")
        ax.set_title(f"{r.dataset} · {r.label}", fontsize=8)
plt.tight_layout(); plt.show()

## Landmark cache (for self-blending)
MediaPipe's face mesh on every **real train/val** crop of every dataset. Images where it finds no face are used
as real only, never blended; coverage should be above ~95%.

In [ ]:
inputs = find_datasets()   # FF++ (01a) and Celeb-DF (01b) from the attached inputs
print("attached:", inputs)
targets = {k: v for k, v in inputs.items() if k in ("FF++", "CelebDF")}
for name in ("FFHQ", "CelebAHQ"):
    if (HQ / name.lower()).is_dir():
        targets[name] = HQ / name.lower()
if "FF++" not in targets:
    print("WARNING: FF++ crops (01a) not attached: no self-blended fakes from FF++ faces")
args = " ".join(f'--dataset "{k}={v}"' for k, v in targets.items())
!TQDM_MININTERVAL=60 python -m src.classifier.landmarks {args} --out /kaggle/working/landmarks --workers 4

In [ ]:
from src.classifier.dataset import list_split
from src.classifier.landmarks import load
from src.classifier.sbi import self_blend

rows = []
for name, root in targets.items():
    path = Path("/kaggle/working/landmarks") / f"{name}.npz"
    n_real = sum(len([i for i in list_split(root, s) if i[2] == 0]) for s in ("train", "val"))
    n_marks = len(np.load(path)["keys"]) if path.exists() else 0
    rows.append({"dataset": name, "real train/val crops": n_real, "with landmarks": n_marks,
                 "coverage": n_marks / max(1, n_real)})
display(pd.DataFrame(rows).style.format({"coverage": "{:.1%}"}))

# A few self-blended examples (fake = the real crop blended with an altered copy of itself)
fig, axes = plt.subplots(len(targets), 6, figsize=(13, 2.3 * len(targets)), squeeze=False)
for row, (name, root) in zip(axes, targets.items()):
    marks = load(Path("/kaggle/working/landmarks") / f"{name}.npz")
    for j, key in enumerate(list(marks)[:3]):
        img = np.asarray(Image.open(Path(root) / key).convert("RGB"))
        fake, mask = self_blend(img, marks[key].astype(np.float32))
        row[2 * j].imshow(img); row[2 * j].set_title(f"{name} real", fontsize=8)
        row[2 * j + 1].imshow(fake); row[2 * j + 1].set_title("self-blended", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/outputs", ignore_errors=True)   # keep only data_hq/ and landmarks/ in the output
!du -sh /kaggle/working/data_hq /kaggle/working/landmarks